# CS Lifeline — Full ML Pipeline
**Wilfred Wilson Osei (22425069) — University of Ghana MSc Computer Science**

This notebook:
1. Loads exported CSVs from the CS Lifeline teacher dashboard
2. Engineers all 12 features
3. Trains LR · DT · RF · GB · BW-RF with 5-fold stratified CV
4. Tests H1 (ablation) and H2 (McNemar + Bonferroni)
5. Computes SHAP values and generates plain-language narratives
6. **Writes predictions directly back to Firebase Firestore** — dashboards update in real time

**Setup:** Upload your exported CSVs using the cell below, then run all cells.


In [ ]:
# Install dependencies
!pip install -q firebase-admin shap scikit-learn pandas numpy matplotlib seaborn mlxtend

In [ ]:
# ── CONFIGURATION ────────────────────────────────────────────────────
# Fill in your Firebase project details and class code

FIREBASE_PROJECT_ID = 'your-project-id'   # from Firebase Console
CLASS_CODE          = '10B'               # the class to process
AT_RISK_THRESHOLD   = 0.50               # accuracy below this = at-risk criterion A
MISC_THRESHOLD      = 3                  # misconception count above this = at-risk criterion C

# Upload your service account key JSON from Firebase Console
# (Project Settings → Service accounts → Generate new private key)
SERVICE_ACCOUNT_KEY = 'service-account-key.json'  # upload this to Colab Files panel
print('Configuration set. Upload your service account key JSON and run the next cell.')

In [ ]:
# ── FIREBASE INITIALISATION ──────────────────────────────────────────
import firebase_admin
from firebase_admin import credentials, firestore

if not firebase_admin._apps:
    cred = credentials.Certificate(SERVICE_ACCOUNT_KEY)
    firebase_admin.initialize_app(cred)

db = firestore.client()
print(f'✓ Firebase connected — project: {FIREBASE_PROJECT_ID}')

In [ ]:
# ── OPTION A: Load directly from Firestore ────────────────────────────
# (Slower for large datasets — use CSV export for better performance)

import pandas as pd
import numpy as np

def load_from_firestore(class_code):
    # Get all student UIDs for this class
    users_ref = db.collection('users')\
        .where('classCode', '==', class_code)\
        .where('role', '==', 'student')
    uids = [doc.id for doc in users_ref.stream()]
    print(f'Found {len(uids)} students in class {class_code}')
    
    # Get all responses for these students (batch by 30 for Firestore 'in' limit)
    responses = []
    for i in range(0, len(uids), 30):
        chunk = uids[i:i+30]
        docs = db.collection('responses')\
            .where('student_uid', 'in', chunk).stream()
        for doc in docs:
            data = doc.to_dict()
            data['response_id'] = doc.id
            if data.get('recorded_at'):
                data['recorded_at'] = data['recorded_at'].isoformat() if hasattr(data['recorded_at'], 'isoformat') else str(data['recorded_at'])
            responses.append(data)
    return pd.DataFrame(responses), uids

responses_df, student_uids = load_from_firestore(CLASS_CODE)
print(f'Loaded {len(responses_df)} responses')
responses_df.head()

In [ ]:
# ── FEATURE ENGINEERING — All 12 derived predictors ──────────────────

MISC_TAGS = [
    'ASSIGN_EQUALITY_CONFUSION', 'VARIABLE_SCOPE_ERROR', 'LOOP_TERMINATION_ERROR',
    'SEQUENTIAL_EXECUTION_ERROR', 'TRACE_PREDICTION_ERROR', 'BOOLEAN_LOGIC_ERROR',
    'DATA_TYPE_CONFUSION', 'PSEUDOCODE_SYNTAX_ERROR', 'ARRAY_INDEX_ERROR', 'RECURSION_MODEL_ERROR'
]

def engineer_features(responses_df):
    df = responses_df.copy()
    df['correct'] = df['correct'].astype(bool)
    
    student_features = []
    
    for uid, grp in df.groupby('student_uid'):
        grp = grp.sort_values('recorded_at') if 'recorded_at' in grp.columns else grp
        n   = len(grp)
        
        # 1. Performance trend slope (rolling accuracy slope)
        acc_series = grp['correct'].astype(int).values
        if len(acc_series) >= 3:
            x = np.arange(len(acc_series))
            slope = np.polyfit(x, acc_series, 1)[0]
        else:
            slope = 0.0
        
        # 2. Consistency metric (std of per-chapter accuracy)
        chapter_acc = grp.groupby('chapter_id')['correct'].mean() if 'chapter_id' in grp.columns else pd.Series([grp['correct'].mean()])
        consistency = 1 - chapter_acc.std() if len(chapter_acc) > 1 else 0.5
        
        # 3. Misconception frequency score
        misc_errors  = grp[~grp['correct'] & grp['misconception_tag'].notna()]
        misc_freq    = len(misc_errors) / n
        
        # 4. Net independent thinking time (mean)
        net_think_ms = grp['net_think_time_ms'].mean() if 'net_think_time_ms' in grp.columns else 0
        
        # 5. Engagement composite
        tab_rate     = grp['tab_switch_count'].mean() if 'tab_switch_count' in grp.columns else 0
        hint_rate    = grp['hint_used'].mean() if 'hint_used' in grp.columns else 0
        changed_rate = grp['changed_answer_after_switch'].mean() if 'changed_answer_after_switch' in grp.columns else 0
        engagement   = (tab_rate + hint_rate + changed_rate) / 3
        
        # 6. Timing anomaly flag (fast wrong answers as fraction)
        fast_wrong   = ((grp['net_think_time_ms'] < 5000) & ~grp['correct']).sum() / n if 'net_think_time_ms' in grp.columns else 0
        
        # 7. Hesitation score (mean time-to-first-selection)
        hesitation   = grp['hesitation_ms'].mean() / 1000 if 'hesitation_ms' in grp.columns else 0
        
        # 8. Answer reversal rate (correct→wrong changes)
        reversal_rate = grp['was_correct_at_any_point'].astype(bool).mean() if 'was_correct_at_any_point' in grp.columns else 0
        false_mastery = ((grp.get('signal_false_mastery', pd.Series([False]*n))).astype(bool)).sum() / n
        
        # 9. Concept inconsistency score
        if 'misconception_tag' in grp.columns:
            tag_acc = grp.groupby('misconception_tag')['correct'].std().mean()
            concept_inconsistency = tag_acc if not np.isnan(tag_acc) else 0
        else:
            concept_inconsistency = 0
        
        # 10. Repeat-question consistency (std of accuracy on same question)
        if 'question_id' in grp.columns:
            rq_std = grp.groupby('question_id')['correct'].std().mean()
            rq_consistency = rq_std if not np.isnan(rq_std) else 0
        else:
            rq_consistency = 0
        
        # 11. Cumulative risk composite (recency-weighted accuracy)
        if len(acc_series) > 1:
            weights      = np.exp(np.linspace(0, 1, len(acc_series)))
            cumulative   = np.average(acc_series, weights=weights)
        else:
            cumulative   = acc_series[0] if len(acc_series) else 0.5
        
        # 12. Hint usage rate (standalone predictor for SHAP attribution)
        hint_usage_rate = hint_rate
        
        # At-risk label (tri-criteria)
        accuracy     = grp['correct'].mean()
        misc_count   = len(misc_errors)
        at_risk      = int(accuracy < AT_RISK_THRESHOLD or misc_count > MISC_THRESHOLD)
        
        student_features.append({
            'student_uid':              uid,
            'question_count':           n,
            'accuracy_pct':             round(accuracy * 100, 1),
            'session_count':            grp['session_id'].nunique() if 'session_id' in grp.columns else 1,
            'avg_tab_switches':         tab_rate,
            
            # 12 engineered features
            'f01_performance_trend_slope':    slope,
            'f02_consistency_metric':         consistency,
            'f03_misconception_freq_score':   misc_freq,
            'f04_net_think_time_ms':          net_think_ms,
            'f05_engagement_composite':       engagement,
            'f06_timing_anomaly_flag':        fast_wrong,
            'f07_hesitation_score':           hesitation,
            'f08_answer_reversal_rate':       reversal_rate,
            'f09_concept_inconsistency':      concept_inconsistency,
            'f10_rq_consistency':             rq_consistency,
            'f11_cumulative_risk_composite':  1 - cumulative,
            'f12_hint_usage_rate':            hint_usage_rate,
            
            'at_risk': at_risk,
        })
    
    return pd.DataFrame(student_features)

features_df = engineer_features(responses_df)
print(f'Feature matrix: {features_df.shape}')
print(f'At-risk: {features_df["at_risk"].sum()} / {len(features_df)} ({features_df["at_risk"].mean()*100:.1f}%)')
features_df.describe()

In [ ]:
# ── MODEL TRAINING — LR · DT · RF · GB · BW-RF ──────────────────────

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score, classification_report
from sklearn.pipeline import Pipeline
import warnings; warnings.filterwarnings('ignore')

FEATURE_COLS = [c for c in features_df.columns if c.startswith('f0') or c.startswith('f1')]
ASSESS_COLS  = ['f01_performance_trend_slope', 'f02_consistency_metric',
                'f03_misconception_freq_score', 'f11_cumulative_risk_composite']
BEHAV_COLS   = [c for c in FEATURE_COLS if c not in ASSESS_COLS]

X_full   = features_df[FEATURE_COLS].fillna(0).values
X_assess = features_df[ASSESS_COLS].fillna(0).values
y        = features_df['at_risk'].values

# Behavioral-Weighted Random Forest — upweights behavioral features during tree construction
class BehavioralWeightedRF(RandomForestClassifier):
    def __init__(self, behavioral_weight=3.0, **kwargs):
        super().__init__(**kwargs)
        self.behavioral_weight = behavioral_weight
    
    def fit(self, X, y, sample_weight=None):
        # Create feature weights: behavioral features get higher sampling probability
        # by repeating them in the feature matrix
        n_assess = len(ASSESS_COLS)
        n_behav  = len(BEHAV_COLS)
        # Tile behavioral columns behavioral_weight times
        X_weighted = np.hstack([
            X[:, :n_assess],
            np.tile(X[:, n_assess:], int(self.behavioral_weight))
        ])
        self._n_original_features = X.shape[1]
        self._n_weighted_features = X_weighted.shape[1]
        return super().fit(X_weighted, y, sample_weight=sample_weight)
    
    def predict(self, X):
        n_assess = len(ASSESS_COLS)
        X_weighted = np.hstack([X[:, :n_assess], np.tile(X[:, n_assess:], int(self.behavioral_weight))])
        return super().predict(X_weighted)
    
    def predict_proba(self, X):
        n_assess = len(ASSESS_COLS)
        X_weighted = np.hstack([X[:, :n_assess], np.tile(X[:, n_assess:], int(self.behavioral_weight))])
        return super().predict_proba(X_weighted)

models = {
    'Logistic Regression': Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(max_iter=1000, random_state=42))]),
    'Decision Tree':       DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=100, random_state=42),
    'BW-RF':               BehavioralWeightedRF(n_estimators=100, behavioral_weight=3, random_state=42),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

print('='*65)
print(f'{'Model':<25} {'Accuracy':>9} {'F1':>8} {'AUC':>8} {'F1 (assess)':>12}')
print('='*65)

for name, model in models.items():
    # Full feature set
    y_pred_full = cross_val_predict(model, X_full, y, cv=cv)
    f1_full     = f1_score(y, y_pred_full, zero_division=0)
    acc_full    = accuracy_score(y, y_pred_full)
    
    # AUC
    try:
        y_prob = cross_val_predict(model, X_full, y, cv=cv, method='predict_proba')[:,1]
        auc    = roc_auc_score(y, y_prob)
    except:
        y_prob = None; auc = float('nan')
    
    # Assessment-only (ablation for H1)
    y_pred_assess = cross_val_predict(model, X_assess, y, cv=cv)
    f1_assess     = f1_score(y, y_pred_assess, zero_division=0)
    f1_lift       = (f1_full - f1_assess) / f1_assess * 100 if f1_assess > 0 else 0
    
    results[name] = {
        'model': model, 'f1_full': f1_full, 'acc': acc_full,
        'auc': auc, 'f1_assess': f1_assess, 'f1_lift': f1_lift,
        'y_pred': y_pred_full, 'y_prob': y_prob
    }
    print(f'{name:<25} {acc_full:>8.3f} {f1_full:>8.3f} {auc:>8.3f} {f1_assess:>10.3f} (+{f1_lift:.1f}%)')

print('='*65)

# H1 TEST
print('\n── H1: Behavioral features improve F1 by ≥10% ──')
for name, r in results.items():
    h1 = '✓ SUPPORTED' if r['f1_lift'] >= 10 else '✗ not supported'
    print(f'  {name}: +{r["f1_lift"]:.1f}% lift → {h1}')

In [ ]:
# ── H2: McNemar + Bonferroni ─────────────────────────────────────────

from mlxtend.evaluate import mcnemar, mcnemar_table

best_interp = max(['Logistic Regression','Decision Tree','Random Forest'], key=lambda k: results[k]['f1_full'])
gb_pred     = results['Gradient Boosting']['y_pred']

comparisons = ['Decision Tree', 'Logistic Regression', 'Random Forest']
alpha_bonf  = 0.05 / len(comparisons)  # Bonferroni correction

print(f'\n── H2: Interpretable models within ≤5% F1 of Gradient Boosting ──')
print(f'Gradient Boosting F1: {results["Gradient Boosting"]["f1_full"]:.3f}')
print(f'Bonferroni α = {alpha_bonf:.4f}\n')

for name in comparisons:
    r      = results[name]
    f1_diff = abs(r['f1_full'] - results['Gradient Boosting']['f1_full'])
    tb     = mcnemar_table(y_target=y, y_model1=r['y_pred'], y_model2=gb_pred)
    chi2, p = mcnemar(ary=tb, corrected=True)
    h2_size = '✓ SUPPORTED' if f1_diff <= 0.05 else '✗ exceeds margin'
    h2_stat = 'not significant' if p > alpha_bonf else 'significant'
    print(f'  {name} vs GB: |ΔF1|={f1_diff:.3f} {h2_size} | p={p:.4f} ({h2_stat})')

In [ ]:
# ── SHAP ANALYSIS — Train final RF on full data ───────────────────────

import shap
import matplotlib.pyplot as plt

# Train final RF on full dataset
rf_final = RandomForestClassifier(n_estimators=100, random_state=42)
rf_final.fit(X_full, y)

explainer   = shap.TreeExplainer(rf_final)
shap_values = explainer.shap_values(X_full)
shap_at_risk = shap_values[1] if isinstance(shap_values, list) else shap_values

# Global feature importance plot
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_at_risk, X_full, feature_names=FEATURE_COLS, show=False, plot_type='bar')
plt.title('SHAP Feature Importance — CS Lifeline Random Forest')
plt.tight_layout()
plt.savefig('shap_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nTop 6 features by mean |SHAP|:')
mean_shap = np.abs(shap_at_risk).mean(0)
for i in np.argsort(mean_shap)[::-1][:6]:
    print(f'  {FEATURE_COLS[i]:<45} {mean_shap[i]:.4f}')

In [ ]:
# ── GENERATE SHAP NARRATIVES per student ─────────────────────────────

FEATURE_READABLE = {
    'f01_performance_trend_slope':    'declining performance trajectory',
    'f02_consistency_metric':         'inconsistent topic coverage',
    'f03_misconception_freq_score':   'high misconception frequency',
    'f04_net_think_time_ms':          'low independent thinking time',
    'f05_engagement_composite':       'low engagement composite',
    'f06_timing_anomaly_flag':        'rapid guessing pattern',
    'f07_hesitation_score':           'elevated hesitation before answering',
    'f08_answer_reversal_rate':       'frequent answer reversals (false mastery signal)',
    'f09_concept_inconsistency':      'inconsistent performance within concept tags',
    'f10_rq_consistency':             'inconsistent on repeated questions',
    'f11_cumulative_risk_composite':  'cumulative at-risk trajectory',
    'f12_hint_usage_rate':            'high hint usage rate',
}

def generate_narrative(uid, risk_category, top_feature_indices, shap_row, risk_prob):
    top_features = [FEATURE_COLS[i] for i in top_feature_indices[:3]]
    readable     = [FEATURE_READABLE.get(f, f.replace('_',' ')) for f in top_features]
    risk_pct     = round(risk_prob * 100)
    
    if risk_category == 'high':
        prefix = f'This student has a {risk_pct}% predicted probability of being at risk. '
        body   = f'The three most influential factors are: {readable[0]}, {readable[1]}, and {readable[2]}. '
        advice = 'Targeted intervention on misconception patterns and formative feedback are recommended before the next summative assessment.'
    elif risk_category == 'borderline':
        prefix = f'This student shows borderline indicators ({risk_pct}% risk probability). '
        body   = f'Key signals include: {readable[0]} and {readable[1]}. '
        advice = 'Monitor closely and consider light-touch check-in on the flagged concept areas.'
    else:
        prefix = f'This student is currently low risk ({risk_pct}% predicted probability). '
        body   = f'Performance is consistent with the strongest signal being {readable[0]}. '
        advice = 'No immediate intervention required. Continue monitoring.'
    
    return prefix + body + advice

# Generate predictions for each student
predictions = []
proba_at_risk = rf_final.predict_proba(X_full)[:, 1]

for i, (_, row) in enumerate(features_df.iterrows()):
    prob  = proba_at_risk[i]
    risk  = 'high' if prob >= 0.70 else 'borderline' if prob >= 0.40 else 'low'
    
    shap_row    = shap_at_risk[i]
    top_idx     = np.argsort(np.abs(shap_row))[::-1][:5]
    top_features= [FEATURE_COLS[j] for j in top_idx]
    
    # Misconception flags from response data
    student_resp = responses_df[responses_df['student_uid'] == row['student_uid']]
    misc_flags   = student_resp[~student_resp['correct'] & student_resp['misconception_tag'].notna()]['misconception_tag'].value_counts().head(3).index.tolist()
    
    narrative = generate_narrative(row['student_uid'], risk, top_idx, shap_row, prob)
    
    predictions.append({
        'uid':               row['student_uid'],
        'class_code':        CLASS_CODE,
        'risk_category':     risk,
        'risk_probability':  float(round(prob, 4)),
        'accuracy_pct':      float(row['accuracy_pct']),
        'question_count':    int(row['question_count']),
        'avg_tab_switches':  float(round(row['avg_tab_switches'], 2)),
        'shap_narrative':    narrative,
        'top_features':      top_features,
        'shap_values':       [round(float(v), 4) for v in shap_row[top_idx].tolist()],
        'misconception_flags': misc_flags,
    })

print(f'Generated {len(predictions)} predictions')
print(f'  High risk:  {sum(1 for p in predictions if p["risk_category"]=="high")}')
print(f'  Borderline: {sum(1 for p in predictions if p["risk_category"]=="borderline")}')
print(f'  Low risk:   {sum(1 for p in predictions if p["risk_category"]=="low")}')

In [ ]:
# ── WRITE PREDICTIONS BACK TO FIRESTORE ──────────────────────────────
# This is the key step — dashboards update in real time via onSnapshot

from google.cloud.firestore_v1 import SERVER_TIMESTAMP

def write_predictions_to_firestore(predictions):
    batch   = db.batch()
    count   = 0
    
    for pred in predictions:
        uid  = pred.pop('uid')  # use UID as document ID
        ref  = db.collection('predictions').document(uid)
        data = {
            **pred,
            'predicted_at': SERVER_TIMESTAMP,
        }
        batch.set(ref, data)  # set (not merge) — full replacement
        count += 1
        pred['uid'] = uid     # restore
        
        # Firestore batch limit is 500
        if count % 400 == 0:
            batch.commit()
            batch = db.batch()
    
    batch.commit()
    print(f'✓ {count} predictions written to Firestore /predictions/')
    print(f'  Teacher dashboard and student dashboards will update automatically.')

write_predictions_to_firestore(predictions)
print('\n🎉 Pipeline complete! Check your CS Lifeline teacher dashboard.')

In [ ]:
# ── MODULE A: Multi-label Misconception Classifier ────────────────────
# One binary classifier per misconception tag

from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.ensemble import RandomForestClassifier

print('Module A — Multi-label Misconception Detection')
print('='*60)

# Build per-tag binary labels
tag_results = {}
for tag in MISC_TAGS:
    # For each student: did they show this misconception?
    tag_labels = []
    for uid in features_df['student_uid']:
        student_resp = responses_df[responses_df['student_uid'] == uid]
        has_tag = ((student_resp['misconception_tag'] == tag) & ~student_resp['correct']).any()
        tag_labels.append(int(has_tag))
    
    y_tag = np.array(tag_labels)
    if y_tag.sum() < 3:
        print(f'  {tag:<40} SKIPPED (insufficient positive examples: {y_tag.sum()})')
        continue
    
    clf    = RandomForestClassifier(n_estimators=50, random_state=42)
    y_pred = cross_val_predict(clf, X_full, y_tag, cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=42))
    
    p = precision_score(y_tag, y_pred, zero_division=0)
    r = recall_score(y_tag, y_pred, zero_division=0)
    f = f1_score(y_tag, y_pred, zero_division=0)
    
    tag_results[tag] = {'precision': p, 'recall': r, 'f1': f}
    print(f'  {tag:<40} P={p:.2f}  R={r:.2f}  F1={f:.2f}')

if tag_results:
    avg_f1 = np.mean([v['f1'] for v in tag_results.values()])
    print(f'\nMacro-averaged F1 across tags: {avg_f1:.3f}')
    print('Benchmark: SANN (Hoq et al., 2025) — Precision 82-92%, Recall 83-98%')

## Pipeline Complete

**What just happened:**
1. ✅ Loaded student response data from Firebase Firestore
2. ✅ Engineered all 12 derived predictors (performance trend, behavioral signals, misconception metrics)
3. ✅ Trained and evaluated LR · DT · RF · GB · BW-RF with 5-fold stratified CV
4. ✅ Tested H1 (ablation: behavioral vs assessment-only features)
5. ✅ Tested H2 (McNemar + Bonferroni: interpretable vs complex models)
6. ✅ Computed SHAP values and generated plain-language teacher narratives
7. ✅ Wrote predictions back to Firebase `/predictions/` collection
8. ✅ Ran Module A multi-label misconception classifier

**Your teacher dashboard and student dashboards will now show:**
- Risk rankings with probability scores
- SHAP-based plain-language explanations per student
- Top misconception flags per student
- Live updates via Firestore `onSnapshot`

**Next run:** Simply re-run all cells after collecting more data. The `predicted_at` timestamp will update automatically.
